# Menu Dataset – Initial Profiling  
CS 513 • Phase I • Task 3

Tables profiled  
* `Dish.csv` (~26 MB)  
* `MenuItem.csv` (~113 MB)  
* `MenuPage.csv` (~4.5 MB)  
* `Menu.csv` (~3.1 MB)


In [4]:
import pandas as pd, duckdb as db
from pathlib import Path
import sys
!pwd
RAW = Path("./data/raw/menus")
dish  = pd.read_csv(RAW / "Dish.csv",      low_memory=False)
item  = pd.read_csv(RAW / "MenuItem.csv",  low_memory=False)
page  = pd.read_csv(RAW / "MenuPage.csv",  low_memory=False)
menu  = pd.read_csv(RAW / "Menu.csv",      low_memory=False)

con = db.connect()
con.register("dish", dish)
con.register("item", item)
con.register("page", page)
con.register("menu", menu)


/Users/ericmodesitt/Desktop/repos/cs513-team65


In [5]:
pd.concat({
    'dish' : dish.isna().mean(),
    'item' : item.isna().mean(),
    'page' : page.isna().mean(),
    'menu' : menu.isna().mean()
}, axis=1).style.background_gradient(cmap="Reds")


,dish,item,page,menu
id,0.000000,0.000000,0.000000,0.000000
name,0.000000,nan,nan,0.817048
description,1.000000,nan,nan,nan
menus_appeared,0.000000,nan,nan,nan
times_appeared,0.000000,nan,nan,nan
first_appeared,0.000000,nan,nan,nan
last_appeared,0.000000,nan,nan,nan
lowest_price,0.067505,nan,nan,nan
highest_price,0.067505,nan,nan,nan
menu_page_id,nan,0.000000,nan,nan


In [7]:
for name, df in [('dish', dish), ('item', item), ('page', page), ('menu', menu)]:
    print(f"{name}:")
    print(df.columns.tolist())
    print("-" * 40)


dish:
['id', 'name', 'description', 'menus_appeared', 'times_appeared', 'first_appeared', 'last_appeared', 'lowest_price', 'highest_price']
----------------------------------------
item:
['id', 'menu_page_id', 'price', 'high_price', 'dish_id', 'created_at', 'updated_at', 'xpos', 'ypos']
----------------------------------------
page:
['id', 'menu_id', 'page_number', 'image_id', 'full_height', 'full_width', 'uuid']
----------------------------------------
menu:
['id', 'name', 'sponsor', 'event', 'venue', 'place', 'physical_description', 'occasion', 'notes', 'call_number', 'keywords', 'language', 'date', 'location', 'location_type', 'currency', 'currency_symbol', 'status', 'page_count', 'dish_count']
----------------------------------------


In [8]:
dup_dish = con.sql("""
    SELECT id AS dish_pk, COUNT(*) n
    FROM dish
    GROUP BY dish_pk
    HAVING n > 1
""").df()

dup_page = con.sql("""
    SELECT id AS page_pk, COUNT(*) n
    FROM page
    GROUP BY page_pk
    HAVING n > 1
""").df()

print("Duplicate dish PKs:", len(dup_dish))
print("Duplicate page PKs:", len(dup_page))
dup_dish.head()


Duplicate dish PKs: 0
Duplicate page PKs: 0


,dish_pk,n


In [10]:
orphan_dish = con.sql("""
    SELECT DISTINCT item.dish_id
    FROM item
    LEFT JOIN dish ON dish.id = item.dish_id
    WHERE dish.id IS NULL
""").df()

orphan_page = con.sql("""
    SELECT DISTINCT item.menu_page_id
    FROM item
    LEFT JOIN page ON page.id = item.menu_page_id
    WHERE page.id IS NULL
""").df()

orphan_menu = con.sql("""
    SELECT DISTINCT page.menu_id
    FROM page
    LEFT JOIN menu ON menu.id = page.menu_id
    WHERE menu.id IS NULL
""").df()

print("Orphan dish_id :", len(orphan_dish))
print("Orphan page_id :", len(orphan_page))
print("Orphan menu_id :", len(orphan_menu))


Orphan dish_id : 4
Orphan page_id : 0
Orphan menu_id : 2254


In [12]:
item['price_num'] = (
    pd.to_numeric(                   # safe converter
        item['price']
            .astype(str)
            .str.replace(r'[^0-9.\-]', '', regex=True),  # keep digits, dot, minus
        errors='coerce'              # put NaN wherever conversion fails
    )
)

# flag crazy prices
outliers = item.query("price_num < 0 or price_num > 500")
print("Extreme price rows:", len(outliers))
outliers[['id', 'dish_id', 'price', 'price_num']].head()


Extreme price rows: 2914


,id,dish_id,price,price_num
6318,7425,1212.0,1000.0,1000.0
410586,431197,99634.0,810.0,810.0
455512,477461,132402.0,630.0,630.0
457731,479829,133877.0,590.0,590.0
457750,479848,133895.0,540.0,540.0


In [13]:
(menu['place']
     .str.strip().str.lower()
     .value_counts()
     .head(20))


place
en route                             293
new york, ny                         198
en route aboard hong kong maru       101
?                                     84
en route aboard ss. kasuga            79
new york                              69
tampa, fl                             63
ss "friedrich der grosse"             55
schnelldampfer "auguste victoria"     54
ny                                    50
ss city of para                       45
ss furst bismarck                     42
ss city of rio de janeiro             41
ss sonoma                             41
st. augustine, fl                     39
ss friedrich der grosse               39
delmonico's, new york, ny             38
tampa,fla.                            37
ss auguste victoria                   34
an bord der "amerika"                 34
Name: count, dtype: int64

In [16]:
examples = pd.concat([
    dup_dish.head(),
    orphan_dish.head(),
    orphan_page.head(),
    outliers.head(),
    menu[menu['place'].str.lower().isin(['nyc', 'new york city'])].head()
])

examples.to_csv("./examples/dirty_rows.csv", index=False)
examples


,dish_pk,n,dish_id,menu_page_id,id,price,high_price,created_at,updated_at,xpos,...,keywords,language,date,location,location_type,currency,currency_symbol,status,page_count,dish_count
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,220797.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,395403.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,329183.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6318,NaN,NaN,1212.0,791.0,7425.0,1000.0,NaN,2011-04-20 20:33:23 UTC,2011-04-20 20:33:23 UTC,0.162857,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
410586,NaN,NaN,99634.0,34065.0,431197.0,810.0,NaN,2011-07-04 15:05:19 UTC,2011-07-04 15:05:19 UTC,0.260000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
455512,NaN,NaN,132402.0,45365.0,477461.0,630.0,NaN,2011-07-22 16:37:45 UTC,2011-07-22 16:38:38 UTC,0.220000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
457731,NaN,NaN,133877.0,45363.0,479829.0,590.0,NaN,2011-07-24 00:18:51 UTC,2011-07-24 00:18:51 UTC,0.127143,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
457750,NaN,NaN,133895.0,45363.0,479848.0,540.0,NaN,2011-07-24 00:25:45 UTC,2011-07-24 00:25:45 UTC,0.144286,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
344,NaN,NaN,NaN,NaN,12898.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1900-02-18,Putnam House,NaN,Dollars,$,complete,2.0,29.0


## 3 · Data-Quality Problems — Initial Scan
The quick scan shows that almost all our core tables are missing important bits of data and don’t link together cleanly. In MenuItem we found just four rows whose dish_id doesn’t exist in the Dish table, but more than 2 × 10^3 pages point to menus that aren’t in Menu at all, so those joins will break. Prices are messy too, about 2.9 k items list impossible numbers (over $500, some over $800), which would ruin any cost analysis. Nearly every metadata column in Menu is-empty (e.g., 81 % of menu names and every keyword is blank), and “New York” appears in 14 different spellings, so city-based grouping will splinter. Cleaning these nulls, orphan keys, crazy prices, and place-name variants is essential before we can trust queries that match dishes to menus and compare prices across cities for our main use case.